In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")


In [2]:
df = pd.read_csv('Student Social Media And Mental Health Impact.csv')

In [ ]:
df = df.drop_duplicates()
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

top_countries = df['Country'].value_counts().index[:10].tolist()

def group_countries(country):
    if country in top_countries:
        return country
    return 'Other'
    
df['Grouped_country'] = df['Country'].apply(group_countries)
df.head()

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score,Grouped_country
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8,Other
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6,Other
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0,Canada
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3,Other
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4,Other


In [4]:
skwewd_col         = ['Study_Hours']
other_numeric_cols = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks", 
                      "Physical_Activity_Hours", "Sleep_Hours_Per_Night"]
ordinal_col        = ['Stress_Level']
normal_col         = ["Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use", "Grouped_country"]

feature_col = skwewd_col + other_numeric_cols + ordinal_col + normal_col

X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


X_train shape: (3498, 12), X_test shape: (1500, 12)


In [5]:
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

plain_numeric_pipeline = Pipeline(steps=[
    ('scale', StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))
])

nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skwewd_col),
    ("Plain_Numeric", plain_numeric_pipeline, other_numeric_cols),
    ('Ordinal', ordinal_pipeline, ordinal_col),
    ('Normal', nominal_pipeline, normal_col)
])


In [7]:
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor(n_neighbors=5, weights='uniform', metric='minkowski', p=2))
])

knn_pipeline.fit(X_train, y_train)

knn_train_preds = knn_pipeline.predict(X_train)
knn_test_preds  = knn_pipeline.predict(X_test)

train_r2 = r2_score(y_train, knn_train_preds)
test_r2  = r2_score(y_test, knn_test_preds)
mae      = mean_absolute_error(y_test, knn_test_preds)
mse      = mean_squared_error(y_test, knn_test_preds)
rmse     = np.sqrt(mse)

print("=" * 45)
print("       BASELINE KNN (k=5) PERFORMANCE")
print("=" * 45)
print(f"Training R² : {train_r2:.4f}")
print(f"Testing R²  : {test_r2:.4f}")
print(f"Testing MAE : {mae:.4f}")
print(f"Testing MSE : {mse:.4f}")
print(f"Testing RMSE: {rmse:.4f}")
print("=" * 45)


       BASELINE KNN (k=5) PERFORMANCE
Training R² : 0.8906
Testing R²  : 0.8400
Testing MAE : 0.3836
Testing MSE : 0.2810
Testing RMSE: 0.5301
